In [ ]:
# Configuration: the tables to copy from bronze to silver.
# To load only a subset, edit this list.
table_names = [
    "product",
    "purchase_order_detail",
    "purchase_order_header",
    "sales_order_detail",
    "sales_territory",
    "vendor",
]
source_schema = "adventure_works.bronze"
target_schema = "adventure_works.silver"

In [ ]:
from pyspark.sql import functions as F

for name in table_names:
    # Silver is built from the bronze tables, not from the raw CSVs.
    df = spark.table(f"{source_schema}.{name}")

    # Light cleaning: trim surrounding whitespace on all string columns.
    # Bronze is already typed, so no casting is needed here.
    string_columns = [c for c, t in df.dtypes if t == "string"]
    for c in string_columns:
        df = df.withColumn(c, F.trim(F.col(c)))

    # Idempotent full overwrite of the silver table.
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{target_schema}.{name}")
    print(name, "rows written:", df.count())

In [ ]:
# Verification: bronze and silver row counts must match for every table.
results = []
for name in table_names:
    bronze_rows = spark.table(f"{source_schema}.{name}").count()
    silver_rows = spark.table(f"{target_schema}.{name}").count()
    results.append((name, bronze_rows, silver_rows, bronze_rows == silver_rows))

verification = spark.createDataFrame(results, ["table", "bronze_rows", "silver_rows", "counts_match"])
display(verification)

assert all(r.counts_match for r in verification.collect()), "Bronze and silver counts disagree - check the overwrite logic in the load cell."
print("Success: all bronze and silver row counts match.")